In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from datasets import load_dataset
from PIL import Image
import numpy as np
import wandb
from tqdm import tqdm

# For metrics & viz:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [8]:
import copy


Load Dataset

In [2]:
ds = load_dataset("HichTala/coco-background", streaming=True)
train = ds["train"].shuffle(seed=42, buffer_size=100)
validation   = ds["validation"]

print(ds)
print(ds["train"].features)
print("num train (for streaming, this might not be precise):")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/103 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/103 [00:00<?, ?it/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['image', 'label'],
        num_shards: 103
    })
    validation: IterableDataset({
        features: ['image', 'label'],
        num_shards: 5
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['airplane', 'apple', 'background', 'backpack', 'banana', 'baseball bat', 'baseball glove', 'bear', 'bed', 'bench', 'bicycle', 'bird', 'boat', 'book', 'bottle', 'bowl', 'broccoli', 'bus', 'cake', 'car', 'carrot', 'cat', 'cell phone', 'chair', 'clock', 'couch', 'cow', 'cup', 'dining table', 'dog', 'donut', 'elephant', 'fire hydrant', 'fork', 'frisbee', 'giraffe', 'hair drier', 'handbag', 'horse', 'hot dog', 'keyboard', 'kite', 'knife', 'laptop', 'microwave', 'motorcycle', 'mouse', 'orange', 'oven', 'parking meter', 'person', 'pizza', 'potted plant', 'refrigerator', 'remote', 'sandwich', 'scissors', 'sheep', 'sink', 'skateboard', 'skis', 'snowboard', 'spoon', 'sports ball', 'stop sign', 'suitcase', 'su

In [3]:
IMAGE_KEY = "image"
LABEL_KEY = "label"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Transforms
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



In [4]:
class HFImageIterableDataset(IterableDataset):
    def __init__(self, hf_split, transform=None, max_samples=None):
        """
        Args:
            hf_split: HuggingFace streaming dataset split
            transform: torchvision transforms
            max_samples: Maximum samples per epoch (important for streaming!)
        """
        self.hf_split = hf_split
        self.transform = transform
        self.max_samples = max_samples

    def __iter__(self):
        count = 0
        try:
            for ex in self.hf_split:
                # Check max samples limit
                if self.max_samples and count >= self.max_samples:
                    break

                try:
                    # Process image
                    img = ex[IMAGE_KEY]
                    if not isinstance(img, Image.Image):
                        img = Image.fromarray(np.array(img))
                    img = img.convert("RGB")

                    # Get label
                    y = int(ex.get(LABEL_KEY, 0))

                    # Apply transforms
                    if self.transform:
                        img = self.transform(img)

                    count += 1
                    yield img, y

                except Exception as e:
                    print(f"Error processing sample {count}: {e}")
                    continue

        except Exception as e:
            print(f"Error in dataset iteration: {e}")
            raise

def get_num_classes(ds_split):
    """Extract number of classes from dataset features"""
    if LABEL_KEY in ds_split.features:
        feature = ds_split.features[LABEL_KEY]
        if hasattr(feature, 'num_classes'):
            return feature.num_classes
    # Fallback: sample the dataset (for streaming)
    print("Sampling dataset to determine num_classes...")
    labels = set()
    count = 0
    for ex in ds_split:
        labels.add(int(ex[LABEL_KEY]))
        count += 1
        if count >= 1000:  # Sample first 1000
            break
    return len(labels)

In [5]:
num_classes = get_num_classes(ds["train"])
print(f"Number of classes: {num_classes}")

Number of classes: 81


In [6]:
config = {
    "batch_size": 64,
    "lr": 1e-4,
    "epochs": 5,
    "num_classes": num_classes,
    "dataset": "HichTala/coco-background",
    "steps_per_epoch": 2000,  # CRITICAL: Limits samples per epoch
    "val_steps": 300,
    "shuffle_buffer": 1000
}

# Initialize wandb
wandb.init(
    project="domain-shift-coco-dota",
    name="resnet50_finetuned_on_coco_background_fixed",
    config=config
)

# Create datasets with LIMITED samples per epoch
train_split = ds["train"].shuffle(seed=42, buffer_size=config["shuffle_buffer"])
val_split = ds["validation"]

# FIXED: Add max_samples to prevent infinite iteration
train_samples_per_epoch = config["steps_per_epoch"] * config["batch_size"]
val_samples_per_epoch = config["val_steps"] * config["batch_size"]

train_ds = HFImageIterableDataset(
    train_split,
    transform=train_tfms,
    max_samples=train_samples_per_epoch
)
val_ds = HFImageIterableDataset(
    val_split,
    transform=val_tfms,
    max_samples=val_samples_per_epoch
)

# Create dataloaders
train_loader = DataLoader(
    train_ds,
    batch_size=config["batch_size"],
    num_workers=2,  # Use 2 workers for better performance
    pin_memory=True,
    prefetch_factor=2
)
val_loader = DataLoader(
    val_ds,
    batch_size=config["batch_size"],
    num_workers=2,
    pin_memory=True
)

# Load model
print(f"Loading ResNet-50 model on {DEVICE}...")
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# --- Start of Head-Only Training Modification ---
# Freeze all parameters in the model backbone
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer and ensure its parameters are trainable
model.fc = nn.Linear(model.fc.in_features, num_classes)
# By default, newly created layers have requires_grad=True, so no need to explicitly set it here.
# --- End of Head-Only Training Modification ---

model = model.to(DEVICE)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
# Only optimize the parameters that require gradients (i.e., the new fc layer)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config["lr"])

# Watch model with wandb
wandb.watch(model, log="all", log_freq=100)

# Validation function
@torch.no_grad()
def run_validation():
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(val_loader, desc="Validation", total=config["val_steps"])
    for step, (x, y) in enumerate(pbar):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

        # Update progress bar
        pbar.set_postfix({
            'loss': running_loss / total,
            'acc': correct / total
        })

    val_loss = running_loss / max(total, 1)
    val_acc = correct / max(total, 1)
    return val_loss, val_acc



# --- IMPORTANT: keep backbone frozen INCLUDING BatchNorm stats ---
# When we freeze weights but call model.train(), BatchNorm running stats would still update.
# For strict feature-extractor behavior, we keep the backbone in eval() and only the head in train().
def set_backbone_eval_keep_head_train(model):
    model.train()
    if hasattr(model, 'fc'):
        model.fc.train()
    for name, module in model.named_children():
        if name != 'fc':
            module.eval()

# Training loop
print("Starting training...")
for epoch in range(config["epochs"]):
    set_backbone_eval_keep_head_train(model)
    running_loss = 0.0
    correct = 0
    total = 0

    # FIXED: Add progress bar to see what's happening
    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{config['epochs']}",
        total=config["steps_per_epoch"]
    )

    for step, (x, y) in enumerate(pbar):
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

        # Update progress bar
        pbar.set_postfix({
            'loss': running_loss / total,
            'acc': correct / total
        })

        # Log to wandb every 50 steps
        if step % 50 == 0:
            wandb.log({
                "train/step_loss": loss.item(),
                "train/step_acc": (preds == y).float().mean().item(),
                "step": epoch * config["steps_per_epoch"] + step
            })

    train_loss = running_loss / max(total, 1)
    train_acc = correct / max(total, 1)

    # Validation
    val_loss, val_acc = run_validation()

    # Log epoch metrics
    wandb.log({
        "epoch": epoch + 1,
        "train/epoch_loss": train_loss,
        "train/epoch_acc": train_acc,
        "val/loss": val_loss,
        "val/acc": val_acc
    })

    print(
        f"\nEpoch {epoch+1}/{config['epochs']} Summary:\n"
        f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.3f}\n"
        f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}\n"
    )

    # Save checkpoint
    checkpoint_path = f"checkpoint_epoch_{epoch+1}.pth"
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
    }, checkpoint_path)
    wandb.save(checkpoint_path)

# Save final model
final_path = "resnet50_coco_background_final.pth"
torch.save(model.state_dict(), final_path)
wandb.save(final_path)

print("Training completed!")
wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mekaarikuchiri (mekaarikuchiri-imt-mines-al-s) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Epoch 5/5:   2%|▏         | 39/2000 [00:34<15:24,  2.12it/s, loss=1.38, acc=0.669]'HTTPSConnectionPool(host='us.gcp.cdn.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/HichTala/coco-background/resolve/01043aba76886745984f789d4e84e6b0ab621189/data/train-00033-of-00103.parquet
Retrying in 1s [Retry 1/5].
Epoch 5/5: 4000it [34:17,  1.94it/s, loss=0.523, acc=0.861]
Validation: 557it [03:54,  2.38it/s, loss=8.33, acc=0.407]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



Epoch 5/5 Summary:
  Train Loss: 0.5225 | Train Acc: 0.861
  Val Loss: 8.3307 | Val Acc: 0.407



wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Training completed!


epoch,▁▃▅▆█
step,▁▁▁▁▁▂▃▃▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▄▅▅▅▆▆▆▆▆▇▆▇▇█
train/epoch_acc,▁▆▇██
train/epoch_loss,█▃▂▁▁
train/step_acc,█▇▁▆▇▁▇▇█▅▇▅█▇██▆█▄▆██▇▆▇█▇▇█▅▄██▇▇▇▆▆▇▅
train/step_loss,▃▅▆▂▁▄█▇▁▃▂▂▁▃▁▁▂▃▃▁▁▃▂▁▁▁▄▂▃▂▁▁▁▂▂▁▁▃▁▁
val/acc,▁▃▆▇█
val/loss,▁▃▄▆█
epoch,5
step,11950
train/epoch_acc,0.86074


Conclusion: froze the backbone, trained the classifier on coco-bg, strong overfitting (the 2 data are not aligned). Since ImageNet is already struggling with coco-bg it might be difficult finetuning later.

In [12]:
import wandb
if wandb.run is None:
    wandb.init(project="domain-shift", name="embedding-extraction", reinit=True)


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [13]:
# =============================
# Domain-shift measurement (COCO vs Target)
# =============================
# This section uses a FROZEN backbone as a feature extractor, then trains a small domain classifier
# on the embeddings to quantify how separable the domains are.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# wandb_enabled = wandb.run is not None  # True if run exists

# if wandb_enabled:
#     wandb.finish()  # stop wandb cleanly before extraction


TARGET_DATASET = "HichTala/dior-background"   # change to "HichTala/dior" or others if needed
TARGET_SPLIT = "train"

# 1) Load a target dataset split (streaming)
try:
    ds_tgt = load_dataset(TARGET_DATASET, streaming=True)
except Exception as e:
    raise RuntimeError(
        f"Failed to load target dataset '{TARGET_DATASET}'. "
        "Update TARGET_DATASET to the correct Hugging Face dataset id. "
        f"Original error: {e}"
    )

tgt_split = ds_tgt[TARGET_SPLIT].shuffle(seed=42, buffer_size=config["shuffle_buffer"])

# 2) Build *evaluation* transforms (no augmentation)
embed_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3) Create loaders for embedding extraction
embed_samples = 5000  # total samples per domain for embedding extraction (adjust to your compute)
embed_bs = 64

coco_embed_ds = HFImageIterableDataset(
    ds["train"].shuffle(seed=42, buffer_size=config["shuffle_buffer"]),
    transform=embed_tfms,
    max_samples=embed_samples
)

tgt_embed_ds = HFImageIterableDataset(
    tgt_split,
    transform=embed_tfms,
    max_samples=embed_samples
)

coco_embed_loader = DataLoader(coco_embed_ds, batch_size=embed_bs, num_workers=2, pin_memory=True)

tgt_embed_loader  = DataLoader(tgt_embed_ds,  batch_size=embed_bs, num_workers=2, pin_memory=True)

# 4) Turn the model into a pure feature extractor: remove classification head
feature_extractor = copy.deepcopy(model).to(DEVICE)
feature_extractor.eval()
feature_extractor.fc = nn.Identity()

@torch.no_grad()
def extract_embeddings(fe, loader, max_batches=None):
    fe.eval()
    all_z = []
    for i, (x, _) in enumerate(tqdm(loader, desc="Extract", leave=False)):
        x = x.to(DEVICE)
        z = fe(x)
        all_z.append(z.detach().cpu())
        if max_batches is not None and (i + 1) >= max_batches:
            break
    return torch.cat(all_z, dim=0).numpy()

Z_coco = extract_embeddings(feature_extractor, coco_embed_loader)
Z_tgt  = extract_embeddings(feature_extractor, tgt_embed_loader)

# 5) Domain classifier on embeddings (0=COCO, 1=Target)

X = np.vstack([Z_coco, Z_tgt])
y = np.hstack([np.zeros(len(Z_coco), dtype=int), np.ones(len(Z_tgt), dtype=int)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba)

print(f"Domain classifier (COCO vs {TARGET_DATASET}) accuracy: {acc:.4f}")
print(f"Domain classifier (COCO vs {TARGET_DATASET}) AUC:      {auc:.4f}")

# Optional: log to wandb
try:
    wandb.log({
        "domain/target": TARGET_DATASET,
        "domain/acc": acc,
        "domain/auc": auc,
        "domain/n_coco": len(Z_coco),
        "domain/n_target": len(Z_tgt)
    })
except Exception:
    pass


Domain classifier (COCO vs HichTala/dior-background) accuracy: 0.9960
Domain classifier (COCO vs HichTala/dior-background) AUC:      0.9998


these dataset are very far apart since AUC is close to 1. LogisticRegression model, after being trained to distinguish between feature embeddings from COCO-background (labeled 0) and Dior-background (labeled 1), can correctly classify the origin of a given feature embedding